# Planning Under Uncertainty: LLM Planners and Their Limits

Planning — generating a sequence of actions to achieve a goal — has a rich history in classical AI. LLMs brought natural language interfaces to planning but introduced new failure modes that classical planners never had.

## Classical Planning: STRIPS and PDDL

Classical AI planning formalized the planning problem as:
- **State space**: a set of logical predicates describing the world
- **Actions**: operators with preconditions and effects
- **Goal**: a desired state to reach
- **Plan**: a sequence of actions that, when applied from the initial state, satisfies the goal

**STRIPS** (Stanford Research Institute Problem Solver, 1971) introduced this formalism. **PDDL** (Planning Domain Definition Language, 1998) became the standard representation for classical planners.

Classical planners are complete and sound — if a plan exists, they find it, and every plan they output is valid. They scale poorly to large state spaces but provide guarantees LLM planners cannot match.

## LLM Planning Capabilities and Failures

LLMs show impressive planning behavior on short-horizon tasks with few constraints. They struggle with:

**Goal maintenance**: forgetting the original goal after several steps, especially in long-horizon tasks.

**Constraint satisfaction**: generating plans that violate constraints stated earlier in the context.

**Dead-end avoidance**: committing to paths that prevent the goal from being achieved.

**State tracking**: in complex environments, losing track of current world state.

Kambhampati et al. (2023) 'Can LLMs Really Reason and Plan?' showed that LLM planning performance degrades sharply as problem complexity increases, and that apparent planning ability on benchmarks often reflects pattern matching on training data rather than genuine reasoning.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Optional

@dataclass
class PlanState:
    variables: dict
    step: int = 0

@dataclass
class PlanAction:
    name: str
    preconditions: dict
    effects: dict

@dataclass
class PlanResult:
    goal: dict
    plan: list
    valid: bool
    failure_reason: str
    steps_executed: int

class PlanValidator:
    def __init__(self, actions: list, initial_state: dict, goal: dict):
        self.actions = {a.name: a for a in actions}
        self.initial = dict(initial_state)
        self.goal = goal

    def check_preconditions(self, state: dict, action: PlanAction) -> bool:
        return all(state.get(k) == v for k, v in action.preconditions.items())

    def apply_action(self, state: dict, action: PlanAction) -> dict:
        new_state = dict(state)
        new_state.update(action.effects)
        return new_state

    def validate_plan(self, plan: list) -> PlanResult:
        state = dict(self.initial)
        for i, action_name in enumerate(plan):
            if action_name not in self.actions:
                return PlanResult(goal=self.goal, plan=plan, valid=False,
                                   failure_reason=f'Unknown action: {action_name}',
                                   steps_executed=i)
            action = self.actions[action_name]
            if not self.check_preconditions(state, action):
                return PlanResult(goal=self.goal, plan=plan, valid=False,
                                   failure_reason=f'Precondition failed at step {i}: {action_name}',
                                   steps_executed=i)
            state = self.apply_action(state, action)
        goal_met = all(state.get(k) == v for k, v in self.goal.items())
        return PlanResult(goal=self.goal, plan=plan, valid=goal_met,
                           failure_reason='' if goal_met else f'Goal not achieved. Final state: {state}',
                           steps_executed=len(plan))

# Blocks world example
actions = [
    PlanAction('pickup_A', {'A_on_table': True, 'hand_empty': True}, {'holding_A': True, 'A_on_table': False, 'hand_empty': False}),
    PlanAction('stack_A_on_B', {'holding_A': True, 'B_clear': True}, {'A_on_B': True, 'holding_A': False, 'hand_empty': True, 'B_clear': False}),
    PlanAction('pickup_B', {'B_on_table': True, 'hand_empty': True, 'B_clear': True}, {'holding_B': True, 'B_on_table': False, 'hand_empty': False}),
    PlanAction('putdown_B', {'holding_B': True}, {'B_on_table': True, 'holding_B': False, 'hand_empty': True, 'B_clear': True}),
]
initial = {'A_on_table': True, 'B_on_table': True, 'hand_empty': True, 'B_clear': True, 'A_on_B': False, 'holding_A': False, 'holding_B': False}
goal = {'A_on_B': True}

validator = PlanValidator(actions, initial, goal)

# Correct plan
correct_plan = ['pickup_A', 'stack_A_on_B']
result = validator.validate_plan(correct_plan)
print(f'Correct plan {correct_plan}: valid={result.valid}')

# LLM-generated incorrect plan (wrong order)
bad_plan = ['stack_A_on_B', 'pickup_A']
result2 = validator.validate_plan(bad_plan)
print(f'Bad plan {bad_plan}: valid={result2.valid}')
print(f'  Failure: {result2.failure_reason}')

## LLM Planner + Constraint Validator

The practical pattern for reliable LLM planning: generate candidate plans with the LLM, validate with a constraint checker, and iterate on failures.

This mirrors the verify-then-refine pattern from notebook 07, but with formal validation rather than model self-critique. The validator is the key — it provides unambiguous feedback that the LLM can act on.

In [ ]:
def llm_planner_with_validation(
    goal_description: str,
    llm_fn: Callable,
    validator: PlanValidator,
    max_attempts: int = 3
) -> dict:
    attempts = []
    for attempt in range(max_attempts):
        if attempt == 0:
            prompt = f'Generate a plan to achieve: {goal_description}\nOutput action names only, one per line.'
        else:
            last = attempts[-1]
            prompt = (
                f'Plan failed: {last["failure_reason"]}\n'
                f'Goal: {goal_description}\nRevise the plan. Output action names only, one per line.'
            )
        raw_plan = llm_fn(prompt)
        plan = [line.strip() for line in raw_plan.strip().split('\n') if line.strip()]
        result = validator.validate_plan(plan)
        attempts.append({
            'attempt': attempt+1, 'plan': plan,
            'valid': result.valid, 'failure_reason': result.failure_reason
        })
        if result.valid:
            break
    return {
        'goal': goal_description,
        'total_attempts': len(attempts),
        'success': attempts[-1]['valid'],
        'final_plan': attempts[-1]['plan'],
        'attempts': attempts,
    }

attempt_counter = [0]
def mock_planner(prompt):
    attempt_counter[0] += 1
    if attempt_counter[0] == 1:
        return 'stack_A_on_B\npickup_A'  # wrong order
    return 'pickup_A\nstack_A_on_B'  # correct

planning_result = llm_planner_with_validation(
    'Put block A on top of block B',
    mock_planner,
    validator,
    max_attempts=3
)
print(f'Planning result:')
print(f'  Success: {planning_result["success"]}')
print(f'  Attempts needed: {planning_result["total_attempts"]}')
print(f'  Final plan: {planning_result["final_plan"]}')
for a in planning_result['attempts']:
    print(f'  Attempt {a["attempt"]}: valid={a["valid"]} | plan={a["plan"]}')
    if not a['valid']:
        print(f'    Failure: {a["failure_reason"][:60]}')

## Planning Complexity and Horizon

LLM planning reliability degrades with:
- **Longer horizons**: 3-5 step plans are generally reliable; 15+ step plans often fail
- **More constraints**: each additional constraint adds complexity for LLMs but not for classical planners
- **State complexity**: more objects and relations increase the chance of tracking errors

For production planning systems, hybrid architectures work best:
- LLM handles natural language understanding and high-level decomposition
- Classical planner or symbolic system handles constraint satisfaction and state tracking
- LLM translates the formal plan back to natural language for the user

This division of labor uses each component's strengths: LLM for language and intent, formal system for correctness.